In [2]:
import pandas as pd
import statsmodels.api as sm

In [3]:
def backward_elimination(y, X, significance_level=0.10):
    """
    Iteratively removes variables with p-values > significance_level.
    
    """
    X = sm.add_constant(X)  # add intercept
    model = sm.OLS(y, X).fit()
    
    while True:
        # Get max p-value
        p_values = model.pvalues
        max_pval = p_values.max()
        worst_var = p_values.idxmax()
        
        # Stop if all p-values are <= significance_level
        if max_pval <= significance_level:
            break
        
        # Do not drop the constant
        if worst_var == "const":
            break
        
        # Drop worst variable
        print(f"Dropping '{worst_var}' (p-value = {max_pval:.4f})")
        X = X.drop(columns=[worst_var])
        
        # Refit model
        model = sm.OLS(y, X).fit()
    
    return model, X

# Benin

In [4]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataRestaurantCPI/Benin_completes"
df = pd.read_excel(x+".xlsx")

In [5]:
# Define X and y
X = df.drop(columns=["Restauration CPI", "Months"])   
X = sm.add_constant(X)       
y = df["Restauration CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.9978)
Dropping 'Maize ($/mt) MAIZE' (p-value = 0.9339)
Dropping 'Rice, Thai A.1 ($/mt) RICE_A1' (p-value = 0.8997)
Dropping 'Fish meal ($/mt) FISH_MEAL' (p-value = 0.8411)
Dropping 'Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.7279)
Dropping 'Sugar, world ($/kg) SUGAR_WLD' (p-value = 0.6985)
Dropping 'Banana, Europe ($/kg) BANANA_EU' (p-value = 0.6302)
Dropping 'Banana, US ($/kg) BANANA_US' (p-value = 0.5295)
Dropping 'Barley ($/mt) BARLEY' (p-value = 0.5284)
Dropping 'Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.5150)
Dropping 'Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.3993)
Dropping 'Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.2539)
Dropping 'Chicken **($/kg) CHICKEN' (p-value = 0.3027)
Dropping 'Palm oil ($/mt) PALM_OIL' (p-value = 0.3060)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.573


In [6]:
# 1. Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.860
Model:                            OLS   Adj. R-squared:                  0.838
Method:                 Least Squares   F-statistic:                     38.72
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           4.02e-86
Time:                        11:35:43   Log-Likelihood:                 879.17
No. Observations:                 293   AIC:                            -1676.
Df Residuals:                     252   BIC:                            -1525.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [7]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Maize ($/mt) MAIZE' (p-value = 0.9755)
Dropping 'Tea, Kolkata ($/kg) TEA_KOLKATA' (p-value = 0.9768)
Dropping 'Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.8049)
Dropping 'Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.7189)
Dropping 'Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.7023)
Dropping 'Palm oil ($/mt) PALM_OIL' (p-value = 0.6580)
Dropping 'Groundnuts ($/mt) GRNUT' (p-value = 0.6694)
Dropping 'Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.7154)
Dropping 'Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.7662)
Dropping 'Banana, Europe ($/kg) BANANA_EU' (p-value = 0.6425)
Dropping 'Barley ($/mt) BARLEY' (p-value = 0.4628)
Dropping 'Sugar, EU ($/kg) SUGAR_EU' (p-value = 0.4834)
Dropping 'Sugar, world ($/kg) SUGAR_WLD' (p-value = 0.3696)
Dropping 'Banana, US ($/kg) BANANA_US' (p-value = 0.2956)
Dropping 'Soybeans ($/mt) SOYBEANS' (p-value = 0.3362)
Dropping 'Groundnut oil **($/mt) GRNUT_OIL' (p-value = 0.3305)
Dropping 'Wheat, US HRW ($/mt) WHEAT_US_HRW' (p-value = 0.1973)
Dro

In [8]:
# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())

                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.570
Model:                            OLS   Adj. R-squared:                  0.504
Method:                 Least Squares   F-statistic:                     8.610
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           1.47e-28
Time:                        11:35:43   Log-Likelihood:                 714.82
No. Observations:                 293   AIC:                            -1350.
Df Residuals:                     253   BIC:                            -1202.
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [9]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.8568)
Dropping 'lag_Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.7869)
Dropping 'lag_Maize ($/mt) MAIZE' (p-value = 0.6258)
Dropping 'lag_Barley ($/mt) BARLEY' (p-value = 0.7682)
Dropping 'lag_Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.6504)
Dropping 'lag_Palm oil ($/mt) PALM_OIL' (p-value = 0.5478)
Dropping 'lag_Fish meal ($/mt) FISH_MEAL' (p-value = 0.3648)
Dropping 'lag_Sugar, US ($/kg) SUGAR_US' (p-value = 0.3525)
Dropping 'lag_Beef **($/kg) BEEF' (p-value = 0.2047)
Dropping 'lag_Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.1632)
Dropping 'lag_Sugar, EU ($/kg) SUGAR_EU' (p-value = 0.2910)
Dropping 'lag_Sugar, world ($/kg) SUGAR_WLD' (p-value = 0.1645)
Dropping 'lag_Banana, Europe ($/kg) BANANA_EU' (p-value = 0.1886)
Dropping 'lag_Banana, US ($/kg) BANANA_US' (p-value = 0.1955)
Dropping 'lag_Soybeans ($/mt) SOYBEANS' (p-value = 0.1114)
Dropping 'lag_Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.1255)

Final mod

In [10]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")



# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.856
Model:                            OLS   Adj. R-squared:                  0.833
Method:                 Least Squares   F-statistic:                     37.48
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           1.24e-84
Time:                        11:35:43   Log-Likelihood:                 875.08
No. Observations:                 293   AIC:                            -1668.
Df Residuals:                     252   BIC:                            -1517.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [11]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.9516)
Dropping 'lag_Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.9370)
Dropping 'lag_Palm kernel oil ($/mt) PLMKRNL_OIL' (p-value = 0.8581)
Dropping 'lag_Barley ($/mt) BARLEY' (p-value = 0.7850)
Dropping 'lag_Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.6917)
Dropping 'lag_Wheat, US HRW ($/mt) WHEAT_US_HRW' (p-value = 0.7032)
Dropping 'lag_Lamb **($/kg) LAMB' (p-value = 0.5871)
Dropping 'lag_Cocoa ($/kg) COCOA' (p-value = 0.5642)
Dropping 'lag_Tea, Kolkata ($/kg) TEA_KOLKATA' (p-value = 0.5659)
Dropping 'lag_Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.8791)
Dropping 'lag_Groundnuts ($/mt) GRNUT' (p-value = 0.6663)
Dropping 'lag_Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.5788)
Dropping 'lag_Orange ($/kg) ORANGE' (p-value = 0.3902)
Dropping 'lag_Groundnut oil **($/mt) GRNUT_OIL' (p-value = 0.3826)
Dropping 'lag_Maize ($/mt) MAIZE' (p-value = 0.4978)
Dropping 'lag_Tea, Colombo ($/kg) TEA_COLOMBO' (p-value = 0.3803)
D

In [12]:
# --- Assuming you already have y from your dataframe ---
# Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# Combine y and its lag, dropping missing values (first row will be NaN)
data = pd.concat([y, y_lagged], axis=1).dropna()

# Define dependent and independent variables
y_current = data[y.name]
X_lag = sm.add_constant(data["lag_y"])  # add intercept

# Fit the OLS model: y_t = α + β*y_{t-1} + ε_t
model_ar1 = sm.OLS(y_current, X_lag).fit()

# Display results
print("\n=== AR(1) model using only y and lag_y ===")
print(model_ar1.summary())


=== AR(1) model using only y and lag_y ===
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.820
Model:                            OLS   Adj. R-squared:                  0.819
Method:                 Least Squares   F-statistic:                     1322.
Date:                Tue, 28 Oct 2025   Prob (F-statistic):          3.21e-110
Time:                        11:35:44   Log-Likelihood:                 841.94
No. Observations:                 293   AIC:                            -1680.
Df Residuals:                     291   BIC:                            -1673.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const   

# Burkina

In [13]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataRestaurantCPI/Burkina_completes"
df = pd.read_excel(x+".xlsx")


In [14]:
# Define X and y
X = df.drop(columns=["Restauration CPI", "Months"])   
X = sm.add_constant(X)       
y = df["Restauration CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Chicken **($/kg) CHICKEN' (p-value = 0.9440)
Dropping 'Banana, Europe ($/kg) BANANA_EU' (p-value = 0.9379)
Dropping 'Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.8157)
Dropping 'Cocoa ($/kg) COCOA' (p-value = 0.8629)
Dropping 'Palm oil ($/mt) PALM_OIL' (p-value = 0.7425)
Dropping 'Rapeseed oil ($/mt) RAPESEED_OIL' (p-value = 0.6566)
Dropping 'Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.6124)
Dropping 'Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.6690)
Dropping 'Groundnut oil **($/mt) GRNUT_OIL' (p-value = 0.6063)
Dropping 'Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.4282)
Dropping 'Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.5173)
Dropping 'Banana, US ($/kg) BANANA_US' (p-value = 0.3155)
Dropping 'Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.2915)
Dropping 'Beef **($/kg) BEEF' (p-value = 0.2458)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                     

In [15]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.798
Model:                            OLS   Adj. R-squared:                  0.766
Method:                 Least Squares   F-statistic:                     24.83
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           1.48e-66
Time:                        11:35:44   Log-Likelihood:                 830.23
No. Observations:                 293   AIC:                            -1578.
Df Residuals:                     252   BIC:                            -1428.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [16]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.9819)
Dropping 'Wheat, US HRW ($/mt) WHEAT_US_HRW' (p-value = 0.9790)
Dropping 'Palm oil ($/mt) PALM_OIL' (p-value = 0.9764)
Dropping 'Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.9306)
Dropping 'Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.9174)
Dropping 'Sugar, world ($/kg) SUGAR_WLD' (p-value = 0.8995)
Dropping 'Banana, Europe ($/kg) BANANA_EU' (p-value = 0.9133)
Dropping 'Barley ($/mt) BARLEY' (p-value = 0.8573)
Dropping 'Coffee, Arabica ($/kg) COFFEE_ARABIC' (p-value = 0.8231)
Dropping 'Chicken **($/kg) CHICKEN' (p-value = 0.7941)
Dropping 'Rapeseed oil ($/mt) RAPESEED_OIL' (p-value = 0.7757)
Dropping 'Beef **($/kg) BEEF' (p-value = 0.7093)
Dropping 'Soybeans ($/mt) SOYBEANS' (p-value = 0.7535)
Dropping 'Banana, US ($/kg) BANANA_US' (p-value = 0.6788)
Dropping 'Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.6808)
Dropping 'Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.7805)
Dropping 'Rice, Thai 25% ($/mt) RICE_25' (p-v

In [17]:

# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.590
Model:                            OLS   Adj. R-squared:                  0.527
Method:                 Least Squares   F-statistic:                     9.348
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           6.61e-31
Time:                        11:35:44   Log-Likelihood:                 726.90
No. Observations:                 293   AIC:                            -1374.
Df Residuals:                     253   BIC:                            -1227.
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [18]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Rapeseed oil ($/mt) RAPESEED_OIL' (p-value = 0.9312)
Dropping 'lag_Palm oil ($/mt) PALM_OIL' (p-value = 0.9148)
Dropping 'lag_Chicken **($/kg) CHICKEN' (p-value = 0.8361)
Dropping 'lag_Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.8570)
Dropping 'lag_Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.7458)
Dropping 'lag_Banana, Europe ($/kg) BANANA_EU' (p-value = 0.7004)
Dropping 'lag_Cocoa ($/kg) COCOA' (p-value = 0.5672)
Dropping 'lag_Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.6332)
Dropping 'lag_Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.5614)
Dropping 'lag_Groundnut oil **($/mt) GRNUT_OIL' (p-value = 0.5593)
Dropping 'lag_Tea, Colombo ($/kg) TEA_COLOMBO' (p-value = 0.5939)
Dropping 'lag_Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.4983)
Dropping 'lag_Sugar, US ($/kg) SUGAR_US' (p-value = 0.1637)
Dropping 'lag_Beef **($/kg) BEEF' (p-value = 0.1296)
Dropping 'lag_Barley ($/mt) BARLEY' (p-value = 0.1585)

Final model summary:
                            OLS Regressio

In [19]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.793
Model:                            OLS   Adj. R-squared:                  0.760
Method:                 Least Squares   F-statistic:                     24.15
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.17e-65
Time:                        11:35:44   Log-Likelihood:                 826.98
No. Observations:                 293   AIC:                            -1572.
Df Residuals:                     252   BIC:                            -1421.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [20]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.9977)
Dropping 'lag_Beef **($/kg) BEEF' (p-value = 0.9685)
Dropping 'lag_Lamb **($/kg) LAMB' (p-value = 0.9355)
Dropping 'lag_Wheat, US HRW ($/mt) WHEAT_US_HRW' (p-value = 0.9209)
Dropping 'lag_Palm oil ($/mt) PALM_OIL' (p-value = 0.8851)
Dropping 'lag_Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.8605)
Dropping 'lag_Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.7595)
Dropping 'lag_Tea, Colombo ($/kg) TEA_COLOMBO' (p-value = 0.6930)
Dropping 'lag_Banana, Europe ($/kg) BANANA_EU' (p-value = 0.6810)
Dropping 'lag_Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.5944)
Dropping 'lag_Chicken **($/kg) CHICKEN' (p-value = 0.6173)
Dropping 'lag_Palm kernel oil ($/mt) PLMKRNL_OIL' (p-value = 0.5345)
Dropping 'lag_Rapeseed oil ($/mt) RAPESEED_OIL' (p-value = 0.5269)
Dropping 'lag_Soybeans ($/mt) SOYBEANS' (p-value = 0.5213)
Dropping 'lag_Banana, US ($/kg) BANANA_US' (p-value = 0.5211)
Dropping 'lag_Sugar, US ($/kg) SUGAR_US' (p-value

In [21]:
# --- Assuming you already have y from your dataframe ---
# Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# Combine y and its lag, dropping missing values (first row will be NaN)
data = pd.concat([y, y_lagged], axis=1).dropna()

# Define dependent and independent variables
y_current = data[y.name]
X_lag = sm.add_constant(data["lag_y"])  # add intercept

# Fit the OLS model: y_t = α + β*y_{t-1} + ε_t
model_ar1 = sm.OLS(y_current, X_lag).fit()

# Display results
print("\n=== AR(1) model using only y and lag_y ===")
print(model_ar1.summary())


=== AR(1) model using only y and lag_y ===
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.753
Model:                            OLS   Adj. R-squared:                  0.752
Method:                 Least Squares   F-statistic:                     888.1
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.08e-90
Time:                        11:35:44   Log-Likelihood:                 801.14
No. Observations:                 293   AIC:                            -1598.
Df Residuals:                     291   BIC:                            -1591.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const   

# Côte d'Ivoire

In [22]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataRestaurantCPI/Ivoire_completes"
df = pd.read_excel(x+".xlsx")


In [23]:
# Define X and y
X = df.drop(columns=["Restauration CPI", "Months"])   
X = sm.add_constant(X)       
y = df["Restauration CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Soybeans ($/mt) SOYBEANS' (p-value = 0.9829)
Dropping 'Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.9803)
Dropping 'Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.9542)
Dropping 'Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.9279)
Dropping 'Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.8175)
Dropping 'Rice, Thai A.1 ($/mt) RICE_A1' (p-value = 0.7012)
Dropping 'Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.6905)
Dropping 'Sorghum ($/mt) SORGHUM' (p-value = 0.5779)
Dropping 'Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.6099)
Dropping 'Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.4019)
Dropping 'Groundnut oil **($/mt) GRNUT_OIL' (p-value = 0.3878)
Dropping 'Banana, Europe ($/kg) BANANA_EU' (p-value = 0.4060)
Dropping 'Rapeseed oil ($/mt) RAPESEED_OIL' (p-value = 0.4439)
Dropping 'Fish meal ($/mt) FISH_MEAL' (p-value = 0.2996)
Dropping 'Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.3644)
Dropping 'Chicken **($/kg) CHICKEN' (p-value = 0.2560)
Dropping 'Orange ($

In [24]:

# 1. Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.833
Model:                            OLS   Adj. R-squared:                  0.806
Method:                 Least Squares   F-statistic:                     31.40
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           1.11e-76
Time:                        11:35:45   Log-Likelihood:                 819.44
No. Observations:                 293   AIC:                            -1557.
Df Residuals:                     252   BIC:                            -1406.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [25]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Palm oil ($/mt) PALM_OIL' (p-value = 0.9647)
Dropping 'Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.9583)
Dropping 'Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.8177)
Dropping 'Groundnuts ($/mt) GRNUT' (p-value = 0.8090)
Dropping 'Rice, Thai A.1 ($/mt) RICE_A1' (p-value = 0.7848)
Dropping 'Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.7188)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.833
Model:                            OLS   Adj. R-squared:                  0.811
Method:                 Least Squares   F-statistic:                     37.77
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.90e-81
Time:                        11:35:45   Log-Likelihood:                 819.26
No. Observations:                 293   AIC:                            -1569.
Df Residuals:                     258   BIC:                   

In [26]:

# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.595
Model:                            OLS   Adj. R-squared:                  0.533
Method:                 Least Squares   F-statistic:                     9.529
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           1.81e-31
Time:                        11:35:45   Log-Likelihood:                 689.73
No. Observations:                 293   AIC:                            -1299.
Df Residuals:                     253   BIC:                            -1152.
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [27]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Fish meal ($/mt) FISH_MEAL' (p-value = 0.9727)
Dropping 'lag_Sorghum ($/mt) SORGHUM' (p-value = 0.9710)
Dropping 'lag_Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.9565)
Dropping 'lag_Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.9562)
Dropping 'lag_Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.8903)
Dropping 'lag_Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.8504)
Dropping 'lag_Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.8453)
Dropping 'lag_Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.8131)
Dropping 'lag_Rice, Thai A.1 ($/mt) RICE_A1' (p-value = 0.7975)
Dropping 'lag_Soybeans ($/mt) SOYBEANS' (p-value = 0.8006)
Dropping 'lag_Groundnut oil **($/mt) GRNUT_OIL' (p-value = 0.4476)
Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.4268)
Dropping 'lag_Coffee, Arabica ($/kg) COFFEE_ARABIC' (p-value = 0.5024)
Dropping 'lag_Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.2942)
Dropping 'lag_Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.3267)
Dropping '

In [28]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.827
Model:                            OLS   Adj. R-squared:                  0.799
Method:                 Least Squares   F-statistic:                     30.09
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           8.28e-75
Time:                        11:35:45   Log-Likelihood:                 814.27
No. Observations:                 293   AIC:                            -1547.
Df Residuals:                     252   BIC:                            -1396.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [29]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Groundnuts ($/mt) GRNUT' (p-value = 0.9857)
Dropping 'lag_Groundnut oil **($/mt) GRNUT_OIL' (p-value = 0.9724)
Dropping 'lag_Rapeseed oil ($/mt) RAPESEED_OIL' (p-value = 0.9510)
Dropping 'lag_Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.9406)
Dropping 'lag_Rice, Thai A.1 ($/mt) RICE_A1' (p-value = 0.9463)
Dropping 'lag_Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.8905)
Dropping 'lag_Beef **($/kg) BEEF' (p-value = 0.7923)
Dropping 'lag_Cocoa ($/kg) COCOA' (p-value = 0.7337)
Dropping 'lag_Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.6235)
Dropping 'lag_Wheat, US HRW ($/mt) WHEAT_US_HRW' (p-value = 0.5936)
Dropping 'lag_Sorghum ($/mt) SORGHUM' (p-value = 0.5837)
Dropping 'lag_Fish meal ($/mt) FISH_MEAL' (p-value = 0.5337)
Dropping 'lag_Palm oil ($/mt) PALM_OIL' (p-value = 0.4283)
Dropping 'lag_Chicken **($/kg) CHICKEN' (p-value = 0.5066)
Dropping 'lag_Palm kernel oil ($/mt) PLMKRNL_OIL' (p-value = 0.3741)
Dropping 'lag_Tea, Kolkata ($/kg) TEA_KOLKATA' (p-value =

In [30]:
# --- Assuming you already have y from your dataframe ---
# Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# Combine y and its lag, dropping missing values (first row will be NaN)
data = pd.concat([y, y_lagged], axis=1).dropna()

# Define dependent and independent variables
y_current = data[y.name]
X_lag = sm.add_constant(data["lag_y"])  # add intercept

# Fit the OLS model: y_t = α + β*y_{t-1} + ε_t
model_ar1 = sm.OLS(y_current, X_lag).fit()

# Display results
print("\n=== AR(1) model using only y and lag_y ===")
print(model_ar1.summary())


=== AR(1) model using only y and lag_y ===
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.791
Model:                            OLS   Adj. R-squared:                  0.791
Method:                 Least Squares   F-statistic:                     1105.
Date:                Tue, 28 Oct 2025   Prob (F-statistic):          4.47e-101
Time:                        11:35:45   Log-Likelihood:                 787.02
No. Observations:                 293   AIC:                            -1570.
Df Residuals:                     291   BIC:                            -1563.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const   

# Guinea

In [31]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataRestaurantCPI/Guinea_completes"
df = pd.read_excel(x+".xlsx")


In [32]:

# Define X and y
X = df.drop(columns=["Restauration CPI", "Months"])   
X = sm.add_constant(X)       
y = df["Restauration CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Orange ($/kg) ORANGE' (p-value = 0.9850)
Dropping 'Banana, Europe ($/kg) BANANA_EU' (p-value = 0.7727)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.712
Model:                            OLS   Adj. R-squared:                  0.665
Method:                 Least Squares   F-statistic:                     15.13
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           4.45e-43
Time:                        11:35:45   Log-Likelihood:                 672.57
No. Observations:                 264   AIC:                            -1269.
Df Residuals:                     226   BIC:                            -1133.
Df Model:                          37                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t   

In [33]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.778
Model:                            OLS   Adj. R-squared:                  0.737
Method:                 Least Squares   F-statistic:                     19.40
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           9.25e-53
Time:                        11:35:45   Log-Likelihood:                 703.51
No. Observations:                 263   AIC:                            -1325.
Df Residuals:                     222   BIC:                            -1179.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [34]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Tea, Colombo ($/kg) TEA_COLOMBO' (p-value = 0.9000)
Dropping 'Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.8608)
Dropping 'Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.8537)
Dropping 'Tea, Kolkata ($/kg) TEA_KOLKATA' (p-value = 0.8579)
Dropping 'Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.8299)
Dropping 'Sugar, US ($/kg) SUGAR_US' (p-value = 0.7772)
Dropping 'Barley ($/mt) BARLEY' (p-value = 0.8218)
Dropping 'Beef **($/kg) BEEF' (p-value = 0.7170)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.777
Model:                            OLS   Adj. R-squared:                  0.746
Method:                 Least Squares   F-statistic:                     25.06
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           4.22e-58
Time:                        11:35:46   Log-Likelihood:                 703.27
No. Observations:                 263  

In [35]:

# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.723
Model:                            OLS   Adj. R-squared:                  0.674
Method:                 Least Squares   F-statistic:                     14.90
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.58e-43
Time:                        11:35:46   Log-Likelihood:                 674.51
No. Observations:                 263   AIC:                            -1269.
Df Residuals:                     223   BIC:                            -1126.
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [36]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())




Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.723
Model:                            OLS   Adj. R-squared:                  0.674
Method:                 Least Squares   F-statistic:                     14.90
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.58e-43
Time:                        11:35:46   Log-Likelihood:                 674.51
No. Observations:                 263   AIC:                            -1269.
Df Residuals:                     223   BIC:                            -1126.
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------

In [37]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.782
Model:                            OLS   Adj. R-squared:                  0.742
Method:                 Least Squares   F-statistic:                     19.86
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           1.32e-53
Time:                        11:35:46   Log-Likelihood:                 705.93
No. Observations:                 263   AIC:                            -1330.
Df Residuals:                     222   BIC:                            -1183.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [38]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Sugar, US ($/kg) SUGAR_US' (p-value = 0.9648)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.782
Model:                            OLS   Adj. R-squared:                  0.743
Method:                 Least Squares   F-statistic:                     20.46
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.90e-54
Time:                        11:35:46   Log-Likelihood:                 705.93
No. Observations:                 263   AIC:                            -1332.
Df Residuals:                     223   BIC:                            -1189.
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
----------------

In [39]:
# --- Assuming you already have y from your dataframe ---
# Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# Combine y and its lag, dropping missing values (first row will be NaN)
data = pd.concat([y, y_lagged], axis=1).dropna()

# Define dependent and independent variables
y_current = data[y.name]
X_lag = sm.add_constant(data["lag_y"])  # add intercept

# Fit the OLS model: y_t = α + β*y_{t-1} + ε_t
model_ar1 = sm.OLS(y_current, X_lag).fit()

# Display results
print("\n=== AR(1) model using only y and lag_y ===")
print(model_ar1.summary())


=== AR(1) model using only y and lag_y ===
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.693
Model:                            OLS   Adj. R-squared:                  0.692
Method:                 Least Squares   F-statistic:                     589.3
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           6.84e-69
Time:                        11:35:46   Log-Likelihood:                 661.17
No. Observations:                 263   AIC:                            -1318.
Df Residuals:                     261   BIC:                            -1311.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const   

# Mali 

In [40]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataRestaurantCPI/Mali_completes"
df = pd.read_excel(x+".xlsx")


In [41]:

# Define X and y
X = df.drop(columns=["Restauration CPI", "Months"])   
X = sm.add_constant(X)       
y = df["Restauration CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Beef **($/kg) BEEF' (p-value = 0.9815)
Dropping 'Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.9761)
Dropping 'Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.9655)
Dropping 'Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.8873)
Dropping 'Lamb **($/kg) LAMB' (p-value = 0.8434)
Dropping 'Maize ($/mt) MAIZE' (p-value = 0.8096)
Dropping 'Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.7377)
Dropping 'Cocoa ($/kg) COCOA' (p-value = 0.6894)
Dropping 'Sugar, US ($/kg) SUGAR_US' (p-value = 0.5594)
Dropping 'Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.6343)
Dropping 'Chicken **($/kg) CHICKEN' (p-value = 0.3722)
Dropping 'Coffee, Arabica ($/kg) COFFEE_ARABIC' (p-value = 0.3620)
Dropping 'Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.3067)
Dropping 'Banana, Europe ($/kg) BANANA_EU' (p-value = 0.2351)
Dropping 'Rice, Thai A.1 ($/mt) RICE_A1' (p-value = 0.3279)
Dropping 'Wheat, US SRW ($/mt) WHEAT_US_SRW' (p-value = 0.2120)
Dropping 'Tea, Colombo ($/kg) TEA_COLOMBO' (p-value = 0.1675)
Droppi

In [42]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.775
Model:                            OLS   Adj. R-squared:                  0.740
Method:                 Least Squares   F-statistic:                     21.74
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           4.65e-61
Time:                        11:35:46   Log-Likelihood:                 744.02
No. Observations:                 293   AIC:                            -1406.
Df Residuals:                     252   BIC:                            -1255.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [43]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.9371)
Dropping 'Beef **($/kg) BEEF' (p-value = 0.9301)
Dropping 'Soybeans ($/mt) SOYBEANS' (p-value = 0.8930)
Dropping 'Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.8604)
Dropping 'Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.8838)
Dropping 'Sugar, US ($/kg) SUGAR_US' (p-value = 0.8525)
Dropping 'Lamb **($/kg) LAMB' (p-value = 0.8305)
Dropping 'Sorghum ($/mt) SORGHUM' (p-value = 0.7453)
Dropping 'Barley ($/mt) BARLEY' (p-value = 0.7712)
Dropping 'Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.7593)
Dropping 'Groundnuts ($/mt) GRNUT' (p-value = 0.7194)
Dropping 'Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.9029)
Dropping 'Tea, Kolkata ($/kg) TEA_KOLKATA' (p-value = 0.6170)
Dropping 'Cocoa ($/kg) COCOA' (p-value = 0.7077)
Dropping 'Tea, Colombo ($/kg) TEA_COLOMBO' (p-value = 0.5044)
Dropping 'Coffee, Arabica ($/kg) COFFEE_ARABIC' (p-value = 0.4997)
Dropping 'Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.8901)

Final model 

In [44]:

# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.610
Model:                            OLS   Adj. R-squared:                  0.550
Method:                 Least Squares   F-statistic:                     10.16
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.20e-33
Time:                        11:35:46   Log-Likelihood:                 663.35
No. Observations:                 293   AIC:                            -1247.
Df Residuals:                     253   BIC:                            -1099.
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [45]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Barley ($/mt) BARLEY' (p-value = 0.8655)
Dropping 'lag_Rice, Thai A.1 ($/mt) RICE_A1' (p-value = 0.8268)
Dropping 'lag_Sorghum ($/mt) SORGHUM' (p-value = 0.7196)
Dropping 'lag_Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.7189)
Dropping 'lag_Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.7039)
Dropping 'lag_Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.5242)
Dropping 'lag_Chicken **($/kg) CHICKEN' (p-value = 0.5478)
Dropping 'lag_Beef **($/kg) BEEF' (p-value = 0.6240)
Dropping 'lag_Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.4786)
Dropping 'lag_Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.4524)
Dropping 'lag_Coffee, Arabica ($/kg) COFFEE_ARABIC' (p-value = 0.4619)
Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.7110)
Dropping 'lag_Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.3184)
Dropping 'lag_Sugar, US ($/kg) SUGAR_US' (p-value = 0.3116)
Dropping 'lag_Banana, Europe ($/kg) BANANA_EU' (p-value = 0.2506)
Dropping 'lag_Tea, Colombo ($/k

In [46]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.774
Model:                            OLS   Adj. R-squared:                  0.738
Method:                 Least Squares   F-statistic:                     21.58
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           9.01e-61
Time:                        11:35:47   Log-Likelihood:                 743.22
No. Observations:                 293   AIC:                            -1404.
Df Residuals:                     252   BIC:                            -1254.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [47]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Tea, Colombo ($/kg) TEA_COLOMBO' (p-value = 0.9595)
Dropping 'lag_Wheat, US SRW ($/mt) WHEAT_US_SRW' (p-value = 0.9528)
Dropping 'lag_Banana, Europe ($/kg) BANANA_EU' (p-value = 0.8221)
Dropping 'lag_Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.7997)
Dropping 'lag_Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.7637)
Dropping 'lag_Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.7360)
Dropping 'lag_Palm oil ($/mt) PALM_OIL' (p-value = 0.7301)
Dropping 'lag_Orange ($/kg) ORANGE' (p-value = 0.6723)
Dropping 'lag_Chicken **($/kg) CHICKEN' (p-value = 0.6634)
Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.6680)
Dropping 'lag_Groundnut oil **($/mt) GRNUT_OIL' (p-value = 0.6809)
Dropping 'lag_Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.5428)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.773
Model:                            OLS   A

In [48]:
# --- Assuming you already have y from your dataframe ---
# Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# Combine y and its lag, dropping missing values (first row will be NaN)
data = pd.concat([y, y_lagged], axis=1).dropna()

# Define dependent and independent variables
y_current = data[y.name]
X_lag = sm.add_constant(data["lag_y"])  # add intercept

# Fit the OLS model: y_t = α + β*y_{t-1} + ε_t
model_ar1 = sm.OLS(y_current, X_lag).fit()

# Display results
print("\n=== AR(1) model using only y and lag_y ===")
print(model_ar1.summary())


=== AR(1) model using only y and lag_y ===
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.720
Model:                            OLS   Adj. R-squared:                  0.719
Method:                 Least Squares   F-statistic:                     749.7
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           1.63e-82
Time:                        11:35:47   Log-Likelihood:                 712.00
No. Observations:                 293   AIC:                            -1420.
Df Residuals:                     291   BIC:                            -1413.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const   

# Niger

In [49]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataRestaurantCPI/Niger_completes"
df = pd.read_excel(x+".xlsx")


In [50]:

# Define X and y
X = df.drop(columns=["Restauration CPI", "Months"])   
X = sm.add_constant(X)       
y = df["Restauration CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Sugar, EU ($/kg) SUGAR_EU' (p-value = 0.8984)
Dropping 'Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.9039)
Dropping 'Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.7963)
Dropping 'Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.8210)
Dropping 'Wheat, US HRW ($/mt) WHEAT_US_HRW' (p-value = 0.8187)
Dropping 'Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.7701)
Dropping 'Coffee, Arabica ($/kg) COFFEE_ARABIC' (p-value = 0.9255)
Dropping 'Sorghum ($/mt) SORGHUM' (p-value = 0.7244)
Dropping 'Orange ($/kg) ORANGE' (p-value = 0.7731)
Dropping 'Banana, US ($/kg) BANANA_US' (p-value = 0.7709)
Dropping 'Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.6891)
Dropping 'Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.6984)
Dropping 'Chicken **($/kg) CHICKEN' (p-value = 0.6358)
Dropping 'Beef **($/kg) BEEF' (p-value = 0.7555)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                

In [51]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.778
Model:                            OLS   Adj. R-squared:                  0.743
Method:                 Least Squares   F-statistic:                     22.14
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           8.25e-62
Time:                        11:35:47   Log-Likelihood:                 496.33
No. Observations:                 293   AIC:                            -910.7
Df Residuals:                     252   BIC:                            -759.8
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [52]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.9897)
Dropping 'Beef **($/kg) BEEF' (p-value = 0.9885)
Dropping 'Orange ($/kg) ORANGE' (p-value = 0.9744)
Dropping 'Sorghum ($/mt) SORGHUM' (p-value = 0.9428)
Dropping 'Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.8902)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.778
Model:                            OLS   Adj. R-squared:                  0.748
Method:                 Least Squares   F-statistic:                     25.80
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.76e-65
Time:                        11:35:47   Log-Likelihood:                 496.31
No. Observations:                 293   AIC:                            -920.6
Df Residuals:                     257   BIC:                            -788.1
Df Model:                          35                                

In [53]:
# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.537
Model:                            OLS   Adj. R-squared:                  0.466
Method:                 Least Squares   F-statistic:                     7.521
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           6.36e-25
Time:                        11:35:47   Log-Likelihood:                 388.29
No. Observations:                 293   AIC:                            -696.6
Df Residuals:                     253   BIC:                            -549.4
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [54]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.9973)
Dropping 'lag_Beef **($/kg) BEEF' (p-value = 0.9035)
Dropping 'lag_Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.7315)
Dropping 'lag_Soybeans ($/mt) SOYBEANS' (p-value = 0.7267)
Dropping 'lag_Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.6531)
Dropping 'lag_Sorghum ($/mt) SORGHUM' (p-value = 0.6095)
Dropping 'lag_Cocoa ($/kg) COCOA' (p-value = 0.6580)
Dropping 'lag_Sugar, EU ($/kg) SUGAR_EU' (p-value = 0.5423)
Dropping 'lag_Chicken **($/kg) CHICKEN' (p-value = 0.4963)
Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.3459)
Dropping 'lag_Orange ($/kg) ORANGE' (p-value = 0.4154)
Dropping 'lag_Sugar, US ($/kg) SUGAR_US' (p-value = 0.3874)
Dropping 'lag_Rapeseed oil ($/mt) RAPESEED_OIL' (p-value = 0.2446)
Dropping 'lag_Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.2486)
Dropping 'lag_Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.2983)
Dropping 'lag_Wheat, US HRW ($/mt) WHEAT_US_HRW' (p-value = 0.3

In [55]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.785
Model:                            OLS   Adj. R-squared:                  0.751
Method:                 Least Squares   F-statistic:                     23.05
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           1.85e-63
Time:                        11:35:47   Log-Likelihood:                 500.93
No. Observations:                 293   AIC:                            -919.9
Df Residuals:                     252   BIC:                            -769.0
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [56]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Lamb **($/kg) LAMB' (p-value = 0.9294)
Dropping 'lag_Tea, Kolkata ($/kg) TEA_KOLKATA' (p-value = 0.9174)
Dropping 'lag_Beef **($/kg) BEEF' (p-value = 0.8984)
Dropping 'lag_Groundnut oil **($/mt) GRNUT_OIL' (p-value = 0.7306)
Dropping 'lag_Fish meal ($/mt) FISH_MEAL' (p-value = 0.6760)
Dropping 'lag_Sugar, world ($/kg) SUGAR_WLD' (p-value = 0.6076)
Dropping 'lag_Soybeans ($/mt) SOYBEANS' (p-value = 0.5930)
Dropping 'lag_Cocoa ($/kg) COCOA' (p-value = 0.6756)
Dropping 'lag_Chicken **($/kg) CHICKEN' (p-value = 0.6032)
Dropping 'lag_Groundnuts ($/mt) GRNUT' (p-value = 0.6044)
Dropping 'lag_Sugar, US ($/kg) SUGAR_US' (p-value = 0.4434)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.784
Model:                            OLS   Adj. R-squared:                  0.760
Method:                 Least Squares   F-statistic:                     32.83
Date:    

# Senegal

In [57]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataRestaurantCPI/Senegal_completes"
df = pd.read_excel(x+".xlsx")


In [58]:

# Define X and y
X = df.drop(columns=["Restauration CPI", "Months"])   
X = sm.add_constant(X)       
y = df["Restauration CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.7901)
Dropping 'Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.7571)
Dropping 'Coffee, Arabica ($/kg) COFFEE_ARABIC' (p-value = 0.7564)
Dropping 'Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.7158)
Dropping 'Orange ($/kg) ORANGE' (p-value = 0.6997)
Dropping 'Soybeans ($/mt) SOYBEANS' (p-value = 0.6464)
Dropping 'Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.6570)
Dropping 'Sorghum ($/mt) SORGHUM' (p-value = 0.5636)
Dropping 'Maize ($/mt) MAIZE' (p-value = 0.4966)
Dropping 'Chicken **($/kg) CHICKEN' (p-value = 0.4293)
Dropping 'Wheat, US HRW ($/mt) WHEAT_US_HRW' (p-value = 0.3416)
Dropping 'Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.3849)
Dropping 'Beef **($/kg) BEEF' (p-value = 0.3416)
Dropping 'Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.3565)
Dropping 'Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.3006)
Dropping 'Groundnuts ($/mt) GRNUT' (p-value = 0.1352)

Final model summary:
                            OLS Regress

In [59]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.865
Model:                            OLS   Adj. R-squared:                  0.843
Method:                 Least Squares   F-statistic:                     40.20
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           7.51e-88
Time:                        11:35:48   Log-Likelihood:                 696.63
No. Observations:                 293   AIC:                            -1311.
Df Residuals:                     252   BIC:                            -1160.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [60]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.8954)
Dropping 'Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.8860)
Dropping 'Beef **($/kg) BEEF' (p-value = 0.8458)
Dropping 'Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.8255)
Dropping 'Wheat, US SRW ($/mt) WHEAT_US_SRW' (p-value = 0.8210)
Dropping 'Orange ($/kg) ORANGE' (p-value = 0.8378)
Dropping 'Barley ($/mt) BARLEY' (p-value = 0.8298)
Dropping 'Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.8539)
Dropping 'Soybeans ($/mt) SOYBEANS' (p-value = 0.7612)
Dropping 'Banana, US ($/kg) BANANA_US' (p-value = 0.6724)
Dropping 'Banana, Europe ($/kg) BANANA_EU' (p-value = 0.7251)
Dropping 'Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.6419)
Dropping 'Groundnuts ($/mt) GRNUT' (p-value = 0.6175)
Dropping 'Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.8082)
Dropping 'Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.7827)
Dropping 'Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.5816)
Dropping 'Coffee, Robusta ($/kg) COFFEE_R

In [61]:
# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.714
Model:                            OLS   Adj. R-squared:                  0.670
Method:                 Least Squares   F-statistic:                     16.18
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           4.08e-49
Time:                        11:35:48   Log-Likelihood:                 587.11
No. Observations:                 293   AIC:                            -1094.
Df Residuals:                     253   BIC:                            -947.0
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [62]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.8876)
Dropping 'lag_Wheat, US HRW ($/mt) WHEAT_US_HRW' (p-value = 0.8753)
Dropping 'lag_Groundnuts ($/mt) GRNUT' (p-value = 0.8792)
Dropping 'lag_Maize ($/mt) MAIZE' (p-value = 0.7375)
Dropping 'lag_Sorghum ($/mt) SORGHUM' (p-value = 0.7120)
Dropping 'lag_Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.6236)
Dropping 'lag_Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.5786)
Dropping 'lag_Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.4832)
Dropping 'lag_Tea, Kolkata ($/kg) TEA_KOLKATA' (p-value = 0.4633)
Dropping 'lag_Chicken **($/kg) CHICKEN' (p-value = 0.5051)
Dropping 'lag_Beef **($/kg) BEEF' (p-value = 0.5334)
Dropping 'lag_Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.4040)
Dropping 'lag_Orange ($/kg) ORANGE' (p-value = 0.3284)
Dropping 'lag_Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.2232)
Dropping 'lag_Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.2852)
Dropping 'lag_Banana, US ($/kg) BANANA_US' (p-value = 0.260

In [63]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.868
Model:                            OLS   Adj. R-squared:                  0.847
Method:                 Least Squares   F-statistic:                     41.42
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           3.04e-89
Time:                        11:35:48   Log-Likelihood:                 700.45
No. Observations:                 293   AIC:                            -1319.
Df Residuals:                     252   BIC:                            -1168.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [64]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Beef **($/kg) BEEF' (p-value = 0.9816)
Dropping 'lag_Sugar, US ($/kg) SUGAR_US' (p-value = 0.9299)
Dropping 'lag_Chicken **($/kg) CHICKEN' (p-value = 0.9024)
Dropping 'lag_Sugar, EU ($/kg) SUGAR_EU' (p-value = 0.8151)
Dropping 'lag_Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.8516)
Dropping 'lag_Fish meal ($/mt) FISH_MEAL' (p-value = 0.7999)
Dropping 'lag_Rice, Thai A.1 ($/mt) RICE_A1' (p-value = 0.7888)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.868
Model:                            OLS   Adj. R-squared:                  0.851
Method:                 Least Squares   F-statistic:                     51.55
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           4.60e-95
Time:                        11:35:48   Log-Likelihood:                 700.30
No. Observations:                 293   AIC:                            -1333.

In [65]:
# --- Assuming you already have y from your dataframe ---
# Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# Combine y and its lag, dropping missing values (first row will be NaN)
data = pd.concat([y, y_lagged], axis=1).dropna()

# Define dependent and independent variables
y_current = data[y.name]
X_lag = sm.add_constant(data["lag_y"])  # add intercept

# Fit the OLS model: y_t = α + β*y_{t-1} + ε_t
model_ar1 = sm.OLS(y_current, X_lag).fit()

# Display results
print("\n=== AR(1) model using only y and lag_y ===")
print(model_ar1.summary())


=== AR(1) model using only y and lag_y ===
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.834
Model:                            OLS   Adj. R-squared:                  0.834
Method:                 Least Squares   F-statistic:                     1465.
Date:                Tue, 28 Oct 2025   Prob (F-statistic):          1.30e-115
Time:                        11:35:48   Log-Likelihood:                 667.16
No. Observations:                 293   AIC:                            -1330.
Df Residuals:                     291   BIC:                            -1323.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const   

# Togo

In [66]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataRestaurantCPI/Togo_completes"
df = pd.read_excel(x+".xlsx")


In [67]:

# Define X and y
X = df.drop(columns=["Restauration CPI", "Months"])   
X = sm.add_constant(X)       
y = df["Restauration CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.7931)
Dropping 'Banana, US ($/kg) BANANA_US' (p-value = 0.7441)
Dropping 'Beef **($/kg) BEEF' (p-value = 0.7308)
Dropping 'Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.6672)
Dropping 'Sorghum ($/mt) SORGHUM' (p-value = 0.6293)
Dropping 'Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.5370)
Dropping 'Sugar, US ($/kg) SUGAR_US' (p-value = 0.5332)
Dropping 'Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.5072)
Dropping 'Banana, Europe ($/kg) BANANA_EU' (p-value = 0.4577)
Dropping 'Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.3166)
Dropping 'Sugar, world ($/kg) SUGAR_WLD' (p-value = 0.3577)
Dropping 'Palm oil ($/mt) PALM_OIL' (p-value = 0.3339)
Dropping 'Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.4030)
Dropping 'Sugar, EU ($/kg) SUGAR_EU' (p-value = 0.3068)
Dropping 'Cocoa ($/kg) COCOA' (p-value = 0.1378)
Dropping 'Rapeseed oil ($/mt) RAPESEED_OIL' (p-value = 0.1346)
Dropping 'Soybeans ($/mt) SOYBEANS' (p-value = 0.4358)

Final

In [68]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.752
Model:                            OLS   Adj. R-squared:                  0.712
Method:                 Least Squares   F-statistic:                     19.09
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           7.04e-56
Time:                        11:35:48   Log-Likelihood:                 708.95
No. Observations:                 293   AIC:                            -1336.
Df Residuals:                     252   BIC:                            -1185.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [69]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Beef **($/kg) BEEF' (p-value = 0.9875)
Dropping 'Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.9102)
Dropping 'Soybeans ($/mt) SOYBEANS' (p-value = 0.9178)
Dropping 'Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.9123)
Dropping 'Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.8720)
Dropping 'Coffee, Arabica ($/kg) COFFEE_ARABIC' (p-value = 0.8982)
Dropping 'Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.7318)
Dropping 'Sugar, world ($/kg) SUGAR_WLD' (p-value = 0.6463)
Dropping 'Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.6115)
Dropping 'Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.5983)
Dropping 'Banana, Europe ($/kg) BANANA_EU' (p-value = 0.5112)
Dropping 'Sugar, US ($/kg) SUGAR_US' (p-value = 0.4282)
Dropping 'Banana, US ($/kg) BANANA_US' (p-value = 0.5220)
Dropping 'Palm oil ($/mt) PALM_OIL' (p-value = 0.2633)
Dropping 'Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.2820)
Dropping 'Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.1799)
Dropping 'Groundnut oil **($/mt) GR

In [70]:
# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.500
Model:                            OLS   Adj. R-squared:                  0.423
Method:                 Least Squares   F-statistic:                     6.481
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.99e-21
Time:                        11:35:49   Log-Likelihood:                 606.25
No. Observations:                 293   AIC:                            -1132.
Df Residuals:                     253   BIC:                            -985.3
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [71]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Sorghum ($/mt) SORGHUM' (p-value = 0.9247)
Dropping 'lag_Palm oil ($/mt) PALM_OIL' (p-value = 0.9252)
Dropping 'lag_Banana, US ($/kg) BANANA_US' (p-value = 0.8741)
Dropping 'lag_Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.8521)
Dropping 'lag_Beef **($/kg) BEEF' (p-value = 0.8446)
Dropping 'lag_Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.8044)
Dropping 'lag_Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.7992)
Dropping 'lag_Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.5499)
Dropping 'lag_Sugar, world ($/kg) SUGAR_WLD' (p-value = 0.6367)
Dropping 'lag_Banana, Europe ($/kg) BANANA_EU' (p-value = 0.3896)
Dropping 'lag_Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.4771)
Dropping 'lag_Sugar, US ($/kg) SUGAR_US' (p-value = 0.3500)
Dropping 'lag_Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.2666)
Dropping 'lag_Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.1636)
Dropping 'lag_Lamb **($/kg) LAMB' (p-value = 0.1590)
Dropping 'lag_Soybeans ($/mt) SOYBEANS' (p-value = 0.1496

In [72]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.731
Model:                            OLS   Adj. R-squared:                  0.689
Method:                 Least Squares   F-statistic:                     17.14
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           9.99e-52
Time:                        11:35:49   Log-Likelihood:                 697.22
No. Observations:                 293   AIC:                            -1312.
Df Residuals:                     252   BIC:                            -1162.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [73]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Soybeans ($/mt) SOYBEANS' (p-value = 0.9994)
Dropping 'lag_Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.9972)
Dropping 'lag_Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.9894)
Dropping 'lag_Sugar, world ($/kg) SUGAR_WLD' (p-value = 0.9310)
Dropping 'lag_Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.9129)
Dropping 'lag_Banana, Europe ($/kg) BANANA_EU' (p-value = 0.8962)
Dropping 'lag_Lamb **($/kg) LAMB' (p-value = 0.7386)
Dropping 'lag_Banana, US ($/kg) BANANA_US' (p-value = 0.7269)
Dropping 'lag_Sugar, US ($/kg) SUGAR_US' (p-value = 0.7331)
Dropping 'lag_Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.7266)
Dropping 'lag_Tea, Colombo ($/kg) TEA_COLOMBO' (p-value = 0.7236)
Dropping 'lag_Beef **($/kg) BEEF' (p-value = 0.6622)
Dropping 'lag_Wheat, US HRW ($/mt) WHEAT_US_HRW' (p-value = 0.6419)
Dropping 'lag_Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.5487)
Dropping 'lag_Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.5816)
Dropping 'lag_Rice, Thai 5% ($/mt) RICE_05' (p-value

In [74]:
# --- Assuming you already have y from your dataframe ---
# Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# Combine y and its lag, dropping missing values (first row will be NaN)
data = pd.concat([y, y_lagged], axis=1).dropna()

# Define dependent and independent variables
y_current = data[y.name]
X_lag = sm.add_constant(data["lag_y"])  # add intercept

# Fit the OLS model: y_t = α + β*y_{t-1} + ε_t
model_ar1 = sm.OLS(y_current, X_lag).fit()

# Display results
print("\n=== AR(1) model using only y and lag_y ===")
print(model_ar1.summary())


=== AR(1) model using only y and lag_y ===
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.687
Model:                            OLS   Adj. R-squared:                  0.686
Method:                 Least Squares   F-statistic:                     637.8
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.58e-75
Time:                        11:35:49   Log-Likelihood:                 674.80
No. Observations:                 293   AIC:                            -1346.
Df Residuals:                     291   BIC:                            -1338.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const   

# UEMOA

In [75]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataRestaurantCPI/UEMOA_completes"
df = pd.read_excel(x+".xlsx")


In [76]:

# Define X and y
X = df.drop(columns=["Restauration CPI", "Months"])   
X = sm.add_constant(X)       
y = df["Restauration CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.9488)
Dropping 'Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.9208)
Dropping 'Soybeans ($/mt) SOYBEANS' (p-value = 0.8901)
Dropping 'Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.8935)
Dropping 'Banana, US ($/kg) BANANA_US' (p-value = 0.8120)
Dropping 'Rapeseed oil ($/mt) RAPESEED_OIL' (p-value = 0.6060)
Dropping 'Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.5893)
Dropping 'Fish meal ($/mt) FISH_MEAL' (p-value = 0.5383)
Dropping 'Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.5811)
Dropping 'Sorghum ($/mt) SORGHUM' (p-value = 0.4836)
Dropping 'Coffee, Arabica ($/kg) COFFEE_ARABIC' (p-value = 0.5794)
Dropping 'Cocoa ($/kg) COCOA' (p-value = 0.6419)
Dropping 'Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.4835)
Dropping 'Palm oil ($/mt) PALM_OIL' (p-value = 0.3354)
Dropping 'Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.2978)
Dropping 'Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.4975)
Dropping 'Barley ($/mt) BARL

In [77]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.808
Model:                            OLS   Adj. R-squared:                  0.777
Method:                 Least Squares   F-statistic:                     26.48
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.81e-69
Time:                        11:35:49   Log-Likelihood:                 835.46
No. Observations:                 293   AIC:                            -1589.
Df Residuals:                     252   BIC:                            -1438.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [78]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Coconut oil ($/mt) COCONUT_OIL' (p-value = 0.8442)
Dropping 'Palm oil ($/mt) PALM_OIL' (p-value = 0.6988)
Dropping 'Coffee, Arabica ($/kg) COFFEE_ARABIC' (p-value = 0.6162)
Dropping 'Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.5755)
Dropping 'Shrimps, Mexican ($/kg) SHRIMP_MEX' (p-value = 0.6093)
Dropping 'Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.5686)
Dropping 'Fish meal ($/mt) FISH_MEAL' (p-value = 0.5397)
Dropping 'Wheat, US SRW ($/mt) WHEAT_US_SRW' (p-value = 0.5327)
Dropping 'Rice, Thai A.1 ($/mt) RICE_A1' (p-value = 0.6734)
Dropping 'Wheat, US HRW ($/mt) WHEAT_US_HRW' (p-value = 0.5902)
Dropping 'Tea, Colombo ($/kg) TEA_COLOMBO' (p-value = 0.4131)
Dropping 'Rice, Thai 25% ($/mt) RICE_25' (p-value = 0.5013)
Dropping 'Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.4587)
Dropping 'Soybean meal ($/mt) SOYBEAN_MEAL' (p-value = 0.7014)
Dropping 'Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.4729)
Dropping 'Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.4516

In [79]:
# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.537
Model:                            OLS   Adj. R-squared:                  0.466
Method:                 Least Squares   F-statistic:                     7.533
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           5.82e-25
Time:                        11:35:49   Log-Likelihood:                 706.74
No. Observations:                 293   AIC:                            -1333.
Df Residuals:                     253   BIC:                            -1186.
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [80]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Cocoa ($/kg) COCOA' (p-value = 0.9154)
Dropping 'lag_Fish meal ($/mt) FISH_MEAL' (p-value = 0.8979)
Dropping 'lag_Sunflower oil ($/mt) SUNFLOWER_OIL' (p-value = 0.8782)
Dropping 'lag_Soybeans ($/mt) SOYBEANS' (p-value = 0.8310)
Dropping 'lag_Palm oil ($/mt) PALM_OIL' (p-value = 0.8022)
Dropping 'lag_Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.8055)
Dropping 'lag_Rice, Thai 5% ($/mt) RICE_05' (p-value = 0.8218)
Dropping 'lag_Banana, Europe ($/kg) BANANA_EU' (p-value = 0.6825)
Dropping 'lag_Sorghum ($/mt) SORGHUM' (p-value = 0.7622)
Dropping 'lag_Tea, Mombasa ($/kg) TEA_MOMBASA' (p-value = 0.6313)
Dropping 'lag_Coffee, Robusta ($/kg) COFFEE_ROBUS' (p-value = 0.4753)
Dropping 'lag_Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.3922)
Dropping 'lag_Banana, US ($/kg) BANANA_US' (p-value = 0.3894)
Dropping 'lag_Rice, Thai A.1 ($/mt) RICE_A1' (p-value = 0.3373)
Dropping 'lag_Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.5828)
Dropping 'lag_Chicken **($/kg) CHICKEN' (p-

In [81]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.802
Model:                            OLS   Adj. R-squared:                  0.770
Method:                 Least Squares   F-statistic:                     25.48
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           1.20e-67
Time:                        11:35:50   Log-Likelihood:                 830.94
No. Observations:                 293   AIC:                            -1580.
Df Residuals:                     252   BIC:                            -1429.
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [82]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Banana, Europe ($/kg) BANANA_EU' (p-value = 0.9945)
Dropping 'lag_Tea, Colombo ($/kg) TEA_COLOMBO' (p-value = 0.9836)
Dropping 'lag_Tea, Kolkata ($/kg) TEA_KOLKATA' (p-value = 0.9851)
Dropping 'lag_Groundnut oil **($/mt) GRNUT_OIL' (p-value = 0.8620)
Dropping 'lag_Soybean oil ($/mt) SOYBEAN_OIL' (p-value = 0.8092)
Dropping 'lag_Fish meal ($/mt) FISH_MEAL' (p-value = 0.9213)
Dropping 'lag_Rice, Viet Namese 5% ($/mt) RICE_05_VNM' (p-value = 0.7886)
Dropping 'lag_Soybeans ($/mt) SOYBEANS' (p-value = 0.7861)
Dropping 'lag_Tea, avg 3 auctions ($/kg) TEA_AVG' (p-value = 0.7402)
Dropping 'lag_Palm kernel oil ($/mt) PLMKRNL_OIL' (p-value = 0.6909)
Dropping 'lag_Chicken **($/kg) CHICKEN' (p-value = 0.6946)
Dropping 'lag_Sorghum ($/mt) SORGHUM' (p-value = 0.5631)
Dropping 'lag_Rice, Thai A.1 ($/mt) RICE_A1' (p-value = 0.6606)
Dropping 'lag_Banana, US ($/kg) BANANA_US' (p-value = 0.5833)
Dropping 'lag_Groundnuts ($/mt) GRNUT' (p-value = 0.5845)
Dropping 'lag_Sugar, US ($/kg) SUGAR_U

In [83]:
# --- Assuming you already have y from your dataframe ---
# Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# Combine y and its lag, dropping missing values (first row will be NaN)
data = pd.concat([y, y_lagged], axis=1).dropna()

# Define dependent and independent variables
y_current = data[y.name]
X_lag = sm.add_constant(data["lag_y"])  # add intercept

# Fit the OLS model: y_t = α + β*y_{t-1} + ε_t
model_ar1 = sm.OLS(y_current, X_lag).fit()

# Display results
print("\n=== AR(1) model using only y and lag_y ===")
print(model_ar1.summary())


=== AR(1) model using only y and lag_y ===
                            OLS Regression Results                            
Dep. Variable:       Restauration CPI   R-squared:                       0.764
Model:                            OLS   Adj. R-squared:                  0.763
Method:                 Least Squares   F-statistic:                     942.3
Date:                Tue, 28 Oct 2025   Prob (F-statistic):           2.97e-93
Time:                        11:35:50   Log-Likelihood:                 805.41
No. Observations:                 293   AIC:                            -1607.
Df Residuals:                     291   BIC:                            -1599.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const   